# 🚢 Titanic ML Pipeline — Complete Research Notebook
### Full Machine Learning System: Preprocessing → 7 Models → Evaluation → Explainability

---

| Item | Detail |
|---|---|
| **Dataset** | Titanic Passenger Manifest — 891 records |
| **Models** | Logistic Regression, SVM, Random Forest, Extra Trees, Gradient Boosting, Decision Tree, Voting Ensemble |
| **Validation** | 5-Fold Stratified Cross-Validation |
| **Metrics** | ROC-AUC, F1-Score, Accuracy |
| **Explainability** | Permutation Importance, Feature Contributions, Decision Paths |

---


## 01 · Environment Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings, io, requests
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import (
    StratifiedKFold, cross_validate, cross_val_score,
    learning_curve, validation_curve, train_test_split
)
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier, VotingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score, precision_score, recall_score,
    confusion_matrix, roc_curve, precision_recall_curve,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

# ── Plot Style ─────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  '#0d1117',
    'axes.facecolor':    '#161b22',
    'axes.edgecolor':    '#30363d',
    'axes.labelcolor':   '#e6edf3',
    'axes.titlecolor':   '#e6edf3',
    'xtick.color':       '#8b949e',
    'ytick.color':       '#8b949e',
    'text.color':        '#e6edf3',
    'grid.color':        '#21262d',
    'grid.linestyle':    '--',
    'grid.alpha':        0.5,
    'legend.facecolor':  '#161b22',
    'legend.edgecolor':  '#30363d',
    'font.family':       'DejaVu Sans',
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.titleweight':  'bold',
})

# Color palette
C = {
    'green':  '#10b981', 'red':    '#ef4444', 'blue':   '#6366f1',
    'yellow': '#f59e0b', 'cyan':   '#06b6d4', 'pink':   '#ec4899',
    'purple': '#8b5cf6', 'orange': '#f97316', 'gray':   '#6b7280',
}
MODEL_COLORS = {
    'Logistic Regression': C['blue'],   'SVM':               C['pink'],
    'Random Forest':       C['green'],  'Extra Trees':        C['cyan'],
    'Gradient Boosting':   C['yellow'], 'Decision Tree':      C['orange'],
    'Voting Ensemble':     C['purple'],
}
PALETTE = [C['blue'], C['pink'], C['green'], C['cyan'], C['yellow'], C['orange'], C['purple']]

print("✅ All imports successful")
print(f"   NumPy    {np.__version__}")
print(f"   Pandas   {pd.__version__}")
import sklearn; print(f"   Sklearn  {sklearn.__version__}")


## 02 · Data Loading & Initial Inspection

In [ ]:
def load_data():
    try:
        df = pd.read_csv('../data/titanic.csv')
        print("✅ Loaded from data/titanic.csv")
    except FileNotFoundError:
        url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
        r   = requests.get(url, timeout=15)
        df  = pd.read_csv(io.StringIO(r.text))
        print("✅ Auto-downloaded from GitHub")
    return df.drop_duplicates()

df = load_data()

print(f"\nShape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nTarget distribution:")
print(df['Survived'].value_counts())
print(f"\nSurvival rate: {df['Survived'].mean()*100:.1f}%")
df.head()


In [ ]:
# Missing value audit
print("=== MISSING VALUES ===")
miss = df.isnull().sum()
miss_pct = (miss / len(df) * 100).round(2)
miss_df = pd.DataFrame({'Missing': miss, 'Percent(%)': miss_pct})
miss_df = miss_df[miss_df['Missing'] > 0].sort_values('Percent(%)', ascending=False)
print(miss_df)

print(f"\n=== DATA TYPES ===")
print(df.dtypes)

print(f"\n=== BASIC STATS ===")
df.describe().round(2)


## 03 · Feature Engineering — 14 Features

In [ ]:
def engineer_features(df):
    d = df.copy()

    # Title extraction from Name
    d['Title'] = d['Name'].str.extract(r',\s*([^.]+)\.')
    rare = d['Title'].value_counts()
    d['Title'] = d['Title'].replace(rare[rare < 10].index, 'Rare')
    title_map = {'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4}
    d['Title_Enc'] = d['Title'].map(title_map).fillna(4).astype(int)

    # Family features
    d['Family_Size']     = d['SibSp'] + d['Parch'] + 1
    d['Is_Alone']        = (d['Family_Size'] == 1).astype(int)
    d['Family_Cat_num']  = pd.cut(d['Family_Size'], bins=[0,1,4,20],
                                   labels=[0,1,2]).astype(float).fillna(0).astype(int)

    # Cabin signal
    d['Has_Cabin']       = d['Cabin'].notna().astype(int)

    # Fare transformation
    d['Log_Fare']        = np.log1p(d['Fare'])
    d['Fare_Per_Person'] = d['Fare'] / d['Family_Size']

    # Encodings
    d['Sex_Enc']         = (d['Sex'] == 'female').astype(int)
    d['Embarked']        = d['Embarked'].fillna('S')
    d['Embarked_Enc']    = d['Embarked'].map({'S':0,'C':1,'Q':2}).fillna(0).astype(int)

    return d

eng_df = engineer_features(df)

FEATURES = [
    'Pclass', 'Sex_Enc', 'Age', 'SibSp', 'Parch', 'Fare',
    'Log_Fare', 'Family_Size', 'Is_Alone', 'Has_Cabin',
    'Embarked_Enc', 'Title_Enc', 'Family_Cat_num', 'Fare_Per_Person'
]
FEATURE_LABELS = {
    'Pclass':'Passenger Class', 'Sex_Enc':'Sex (Female=1)', 'Age':'Age',
    'SibSp':'Siblings/Spouse', 'Parch':'Parents/Children', 'Fare':'Fare (£)',
    'Log_Fare':'Log(Fare)', 'Family_Size':'Family Size', 'Is_Alone':'Is Alone',
    'Has_Cabin':'Has Cabin', 'Embarked_Enc':'Embarked Port',
    'Title_Enc':'Title', 'Family_Cat_num':'Family Category',
    'Fare_Per_Person':'Fare Per Person'
}

print(f"✅ Engineered {len(FEATURES)} features:")
for f in FEATURES:
    print(f"   {f:22s} → {FEATURE_LABELS[f]}")


## 04 · Preprocessing — Imputation, Scaling

In [ ]:
def prepare_xy(df_eng):
    X = df_eng[FEATURES].copy()

    # Age: group-based median imputation (Pclass × Sex)
    print("=== AGE IMPUTATION ===")
    for pclass in [1, 2, 3]:
        for sex in ['male', 'female']:
            sex_enc     = 1 if sex == 'female' else 0
            mask_all    = (df_eng['Pclass'] == pclass) & (df_eng['Sex'] == sex)
            mask_valid  = mask_all & df_eng['Age'].notna()
            median_age  = df_eng.loc[mask_valid, 'Age'].median()
            mask_miss   = mask_all & df_eng['Age'].isna()
            n_imputed   = mask_miss.sum()
            if n_imputed > 0:
                X.loc[mask_miss, 'Age'] = median_age
                print(f"  Pclass={pclass} {sex:6s}: median={median_age:.1f} → {n_imputed} imputed")

    X['Age']             = X['Age'].fillna(X['Age'].median())
    X['Fare_Per_Person'] = X['Fare_Per_Person'].fillna(X['Fare_Per_Person'].median())

    print(f"\nRemaining NaN: {X.isna().sum().sum()}")
    y = df_eng['Survived'].copy()
    return X.astype(float), y

X, y = prepare_xy(eng_df)

# Scale
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

print(f"\n✅ X shape: {X.shape}")
print(f"   y distribution: {y.value_counts().to_dict()}")
print(f"   X_sc mean ≈ {X_sc.mean():.6f}, std ≈ {X_sc.std():.6f}")


## 05 · Preprocessing Visualization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Preprocessing Analysis', fontsize=16, fontweight='bold', y=0.98)

# 1. Missing values
miss_vals = df[FEATURES].isnull().sum()
miss_vals = miss_vals[miss_vals > 0]
axes[0,0].bar(miss_vals.index, miss_vals.values / len(df) * 100,
              color=C['red'], alpha=0.85, edgecolor=C['red'], linewidth=0.5)
axes[0,0].set_title('Missing Values (%)')
axes[0,0].set_ylabel('Missing %')
for i, v in enumerate(miss_vals.values / len(df) * 100):
    axes[0,0].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=10, color=C['red'])

# 2. Target balance
vc = y.value_counts()
colors_pie = [C['red'], C['green']]
wedges, texts, autotexts = axes[0,1].pie(
    vc.values, labels=['Not Survived', 'Survived'],
    autopct='%1.1f%%', colors=colors_pie,
    startangle=90, pctdistance=0.75,
    wedgeprops=dict(edgecolor='#0d1117', linewidth=2)
)
for t in autotexts: t.set_color('white'); t.set_fontsize(12)
axes[0,1].set_title('Class Balance (Imbalance = 38.4%)')

# 3. Age before/after imputation
axes[0,2].hist(df['Age'].dropna(), bins=35, color=C['red'], alpha=0.6,
               label=f'Before ({df["Age"].isna().sum()} NaN)', edgecolor='none')
axes[0,2].hist(X['Age'], bins=35, color=C['green'], alpha=0.5,
               label='After (0 NaN)', edgecolor='none')
axes[0,2].axvline(df['Age'].median(), color='white', linestyle='--', alpha=0.5)
axes[0,2].set_title('Age: Before vs After Imputation')
axes[0,2].set_xlabel('Age')
axes[0,2].legend()

# 4. Fare distribution + log transform
axes[1,0].hist(X['Fare'], bins=50, color=C['purple'], alpha=0.85, edgecolor='none')
axes[1,0].set_title(f'Fare Distribution (skew={X["Fare"].skew():.2f})')
axes[1,0].set_xlabel('Fare (£)')

axes[1,1].hist(X['Log_Fare'], bins=50, color=C['green'], alpha=0.85, edgecolor='none')
axes[1,1].set_title(f'Log(Fare) Distribution (skew={X["Log_Fare"].skew():.2f})')
axes[1,1].set_xlabel('log(1 + Fare)')

# 5. Feature correlation with target
corr_with_target = pd.concat([X, y], axis=1).corr()['Survived'].drop('Survived')
corr_sorted = corr_with_target.sort_values()
colors_corr = [C['red'] if v < 0 else C['green'] for v in corr_sorted]
axes[1,2].barh([FEATURE_LABELS.get(f,f) for f in corr_sorted.index],
                corr_sorted.values, color=colors_corr, alpha=0.85)
axes[1,2].axvline(0, color='white', linewidth=1)
axes[1,2].set_title('Pearson Correlation with Survived')
axes[1,2].set_xlabel('Correlation r')

plt.tight_layout()
plt.savefig('plots/05_preprocessing.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print("✅ Saved: plots/05_preprocessing.png")


## 06 · Model Definitions — 7 Classifiers

In [ ]:
def get_models():
    lr = LogisticRegression(
        max_iter=1000, C=1.0, class_weight='balanced',
        solver='lbfgs', random_state=42
    )
    svm = SVC(
        probability=True, kernel='rbf', C=1.0, gamma='scale',
        class_weight='balanced', random_state=42
    )
    rf = RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_split=4,
        min_samples_leaf=2, class_weight='balanced',
        random_state=42, n_jobs=-1
    )
    et = ExtraTreesClassifier(
        n_estimators=200, max_depth=8, min_samples_split=4,
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    gb = GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, random_state=42
    )
    dt = DecisionTreeClassifier(
        max_depth=6, min_samples_split=4, min_samples_leaf=2,
        class_weight='balanced', random_state=42
    )
    ensemble = VotingClassifier(
        estimators=[
            ('lr', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)),
            ('rf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)),
            ('gb', GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, random_state=42))
        ],
        voting='soft'
    )
    return {
        'Logistic Regression': lr, 'SVM': svm,
        'Random Forest': rf,       'Extra Trees': et,
        'Gradient Boosting': gb,   'Decision Tree': dt,
        'Voting Ensemble': ensemble,
    }

models = get_models()
print("Model Configurations:")
print("=" * 60)
for name, model in models.items():
    params = model.get_params()
    key_params = {k: v for k, v in params.items()
                  if k in ['n_estimators','max_depth','C','learning_rate',
                           'max_iter','kernel','class_weight','voting']}
    print(f"  {name:22s} → {key_params}")


## 07 · 5-Fold Stratified Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}
trained_models = {}

print("Running 5-Fold Stratified CV on all 7 models...")
print("=" * 70)

for name, model in models.items():
    res = cross_validate(
        model, X_sc, y.values, cv=cv,
        scoring=['accuracy', 'roc_auc', 'f1'],
        return_train_score=True,
        return_estimator=True,
        n_jobs=-1
    )
    cv_results[name] = res

    # Refit on full data
    model.fit(X_sc, y.values)
    trained_models[name] = model

    auc_m = res['test_roc_auc'].mean()
    auc_s = res['test_roc_auc'].std()
    acc_m = res['test_accuracy'].mean()
    f1_m  = res['test_f1'].mean()
    gap   = res['train_roc_auc'].mean() - auc_m

    star  = " ★ BEST" if name == max(cv_results,
              key=lambda k: cv_results[k]['test_roc_auc'].mean()) else ""
    print(f"  {name:22s}  AUC={auc_m:.4f}±{auc_s:.4f}  "
          f"Acc={acc_m:.4f}  F1={f1_m:.4f}  Gap={gap:.4f}{star}")

best_model_name = max(cv_results, key=lambda k: cv_results[k]['test_roc_auc'].mean())
print(f"\n🏆 Best Model: {best_model_name}")
print(f"   AUC = {cv_results[best_model_name]['test_roc_auc'].mean()*100:.2f}%")


## 08 · Results Summary Table

In [ ]:
rows = []
for name, res in cv_results.items():
    rows.append({
        'Model':            name,
        'ROC-AUC (mean)':   f"{res['test_roc_auc'].mean()*100:.2f}%",
        'ROC-AUC (±std)':   f"±{res['test_roc_auc'].std()*100:.2f}%",
        'Accuracy':         f"{res['test_accuracy'].mean()*100:.2f}%",
        'F1-Score':         f"{res['test_f1'].mean()*100:.2f}%",
        'Train AUC':        f"{res['train_roc_auc'].mean()*100:.2f}%",
        'Train-CV Gap':     f"{(res['train_roc_auc'].mean()-res['test_roc_auc'].mean())*100:.2f}%",
        '🏆':              '★' if name == best_model_name else '',
    })

results_df = pd.DataFrame(rows)
results_df = results_df.sort_values('ROC-AUC (mean)', ascending=False).reset_index(drop=True)

# Style
def highlight_best(s):
    return ['background-color: rgba(16,185,129,0.2); font-weight:bold'
            if v == '★' else '' for v in s]

styled = (results_df.style
          .apply(highlight_best, subset=['🏆'])
          .set_caption("5-Fold Stratified CV Results — All Models"))
print(results_df.to_string(index=False))


## 09 · Model Comparison Visualizations

In [ ]:
names_sorted = sorted(cv_results.keys(),
    key=lambda k: cv_results[k]['test_roc_auc'].mean(), reverse=True)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('5-Fold Cross-Validation Results — All Models', fontsize=16,
             fontweight='bold', y=1.01)

# 1. ROC-AUC comparison
aucs  = [cv_results[n]['test_roc_auc'].mean() for n in names_sorted]
stds  = [cv_results[n]['test_roc_auc'].std()  for n in names_sorted]
colors_bar = [MODEL_COLORS[n] for n in names_sorted]
bars = axes[0,0].barh(names_sorted[::-1], aucs[::-1],
                       xerr=stds[::-1], color=colors_bar[::-1],
                       alpha=0.88, capsize=5,
                       error_kw=dict(ecolor='white', capthick=2))
axes[0,0].axvline(0.5, color=C['red'], linestyle='--', alpha=0.6, label='Random')
axes[0,0].set_xlabel('ROC-AUC')
axes[0,0].set_title('ROC-AUC (5-Fold CV mean ± std)')
for bar, val, std in zip(bars.patches, aucs[::-1], stds[::-1]):
    axes[0,0].text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                    f'{val*100:.2f}%', va='center', fontsize=9)
axes[0,0].legend()

# 2. All metrics grouped bar
x = np.arange(len(names_sorted))
w = 0.26
metric_sets = [
    ('test_roc_auc', 'ROC-AUC',  C['blue']),
    ('test_accuracy', 'Accuracy', C['green']),
    ('test_f1',       'F1-Score', C['yellow']),
]
for i, (metric, label, color) in enumerate(metric_sets):
    vals = [cv_results[n][metric].mean() for n in names_sorted]
    axes[0,1].bar(x + i*w - w, vals, w, label=label, color=color, alpha=0.85)
axes[0,1].set_xticks(x)
axes[0,1].set_xticklabels([n.replace(' ', '\n') for n in names_sorted], fontsize=9)
axes[0,1].set_ylabel('Score')
axes[0,1].set_title('All Metrics Comparison')
axes[0,1].legend()
axes[0,1].set_ylim(0, 1.05)

# 3. Bias-variance gap
gaps = [(cv_results[n]['train_roc_auc'].mean() -
         cv_results[n]['test_roc_auc'].mean()) for n in names_sorted]
colors_gap = [C['yellow'] if g < 0.05 else C['orange'] if g < 0.12 else C['red']
              for g in gaps]
axes[1,0].bar(names_sorted, gaps, color=colors_gap, alpha=0.88)
axes[1,0].axhline(0.05, color=C['yellow'], linestyle='--', alpha=0.7, label='Slight overfit')
axes[1,0].axhline(0.12, color=C['red'],    linestyle='--', alpha=0.7, label='Strong overfit')
axes[1,0].set_xticklabels([n.replace(' ','\n') for n in names_sorted], fontsize=9)
axes[1,0].set_ylabel('Train AUC − CV AUC')
axes[1,0].set_title('Bias-Variance Gap (lower = better generalization)')
axes[1,0].legend()

# 4. Fold-by-fold for best model
best_folds = cv_results[best_model_name]['test_roc_auc']
train_folds = cv_results[best_model_name]['train_roc_auc']
fold_labels = [f'Fold {i+1}' for i in range(5)]
axes[1,1].plot(fold_labels, train_folds, 'o--', color=C['gray'],
               linewidth=2, markersize=9, label='Train AUC')
axes[1,1].plot(fold_labels, best_folds, 'D-', color=MODEL_COLORS[best_model_name],
               linewidth=3, markersize=11, label='CV AUC')
axes[1,1].axhline(best_folds.mean(), color=C['green'], linestyle=':', linewidth=2,
                   label=f'Mean={best_folds.mean()*100:.2f}%')
axes[1,1].fill_between(fold_labels,
    [best_folds.mean() - best_folds.std()] * 5,
    [best_folds.mean() + best_folds.std()] * 5,
    alpha=0.15, color=MODEL_COLORS[best_model_name])
axes[1,1].set_ylim(0.5, 1.05)
axes[1,1].set_ylabel('ROC-AUC')
axes[1,1].set_title(f'{best_model_name} — Fold-by-Fold AUC')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('plots/09_model_comparison.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print("✅ Saved: plots/09_model_comparison.png")


## 10 · ROC Curves & AUC Analysis

In [ ]:
# Get prediction probabilities
probas = {}
for name, model in trained_models.items():
    probas[name] = model.predict_proba(X_sc)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('ROC & Precision-Recall Curves', fontsize=15, fontweight='bold')

# ROC curves
axes[0].plot([0,1],[0,1], '--', color=C['gray'], linewidth=1.5, label='Random (AUC=0.50)')
for name in names_sorted:
    fpr, tpr, _ = roc_curve(y.values, probas[name])
    auc = roc_auc_score(y.values, probas[name])
    is_best = name == best_model_name
    lw  = 3.5 if is_best else 1.8
    lbl = f"{'★ ' if is_best else ''}{name} ({auc:.3f})"
    axes[0].plot(fpr, tpr, linewidth=lw, color=MODEL_COLORS[name],
                  label=lbl, alpha=1.0 if is_best else 0.8)
    if is_best:
        axes[0].fill_between(fpr, tpr, alpha=0.08, color=MODEL_COLORS[name])

axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves (Full Training Set)')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1.02)

# PR curves
baseline = y.mean()
axes[1].axhline(baseline, color=C['gray'], linestyle='--',
                 label=f'Baseline (Prevalence={baseline:.2f})')
for name in names_sorted:
    prec, rec, _ = precision_recall_curve(y.values, probas[name])
    is_best = name == best_model_name
    axes[1].plot(rec, prec, linewidth=3.5 if is_best else 1.8,
                  color=MODEL_COLORS[name], label=name,
                  alpha=1.0 if is_best else 0.8)

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves')
axes[1].legend(loc='upper right', fontsize=9)
axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 1.02)

plt.tight_layout()
plt.savefig('plots/10_roc_pr_curves.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print("✅ Saved: plots/10_roc_pr_curves.png")


## 11 · Confusion Matrices — All Models

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
fig.suptitle('Confusion Matrices (threshold=0.5, full training set)', fontsize=15,
             fontweight='bold')
axes = axes.flatten()

for i, name in enumerate(names_sorted):
    preds = (probas[name] >= 0.5).astype(int)
    cm    = confusion_matrix(y.values, preds)
    tn, fp, fn, tp = cm.ravel()
    acc  = accuracy_score(y.values, preds)
    auc  = roc_auc_score(y.values, probas[name])
    f1   = f1_score(y.values, preds)
    color = MODEL_COLORS[name]

    sns.heatmap(cm, annot=True, fmt='d', ax=axes[i],
                cmap=sns.light_palette(color, as_cmap=True),
                xticklabels=['Pred:Not', 'Pred:Surv'],
                yticklabels=['True:Not', 'True:Surv'],
                linewidths=2, linecolor='#0d1117',
                annot_kws={'size':16, 'weight':'bold'})
    star = ' ★' if name == best_model_name else ''
    axes[i].set_title(
        f"{name}{star}\nAUC={auc:.3f}  Acc={acc:.3f}  F1={f1:.3f}",
        fontsize=10
    )

axes[-1].axis('off')
plt.tight_layout()
plt.savefig('plots/11_confusion_matrices.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print("✅ Saved: plots/11_confusion_matrices.png")


## 12 · Feature Importance — All Models

In [ ]:
feat_labels = [FEATURE_LABELS.get(f, f) for f in FEATURES]

# Collect importances
importances = {}
for name in ['Random Forest', 'Extra Trees', 'Gradient Boosting', 'Decision Tree']:
    importances[name] = trained_models[name].feature_importances_
importances['Logistic Regression'] = np.abs(trained_models['Logistic Regression'].coef_[0])

# Normalize
norm_imp = {}
for name, imp in importances.items():
    norm_imp[name] = imp / (imp.sum() + 1e-9)

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Feature Importance — All Models (Normalized)', fontsize=15,
             fontweight='bold')
axes = axes.flatten()

for i, (name, imp) in enumerate(norm_imp.items()):
    idx = np.argsort(imp)
    colors_bar = [MODEL_COLORS[name]] * len(FEATURES)
    bars = axes[i].barh([feat_labels[j] for j in idx], imp[idx],
                         color=MODEL_COLORS[name], alpha=0.85)
    # Highlight top 3
    top3 = np.argsort(imp)[-3:]
    for bar_idx, bar in enumerate(bars.patches):
        if idx[bar_idx] in top3:
            bar.set_alpha(1.0)
            bar.set_edgecolor('white')
            bar.set_linewidth(1.5)
    axes[i].set_title(name)
    axes[i].set_xlabel('Normalized Importance')

# Combined average
avg_imp = np.mean(list(norm_imp.values()), axis=0)
idx = np.argsort(avg_imp)
axes[-1].barh([feat_labels[j] for j in idx], avg_imp[idx],
               color=PALETTE[:len(FEATURES)], alpha=0.85)
axes[-1].set_title('Average — All Models Combined')
axes[-1].set_xlabel('Avg Normalized Importance')

plt.tight_layout()
plt.savefig('plots/12_feature_importance.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print("✅ Saved: plots/12_feature_importance.png")

# Print top 5
print("\n=== TOP 5 FEATURES (Average) ===")
top5_idx = np.argsort(avg_imp)[-5:][::-1]
for rank, i in enumerate(top5_idx, 1):
    print(f"  {rank}. {feat_labels[i]:25s} {avg_imp[i]:.4f}")


## 13 · Permutation Importance — Model-Agnostic Explainability

> **What is Permutation Importance?**
> For each feature, we randomly shuffle its values and measure how much the model's AUC drops.
> A large drop = feature is important. A small drop = feature is not critical.
> Unlike built-in importance, this works for **any model** including SVM and Logistic Regression.


In [ ]:
print("Computing Permutation Importance (n_repeats=15)...")
print("This may take 1-2 minutes...\n")

perm_results = {}
for name in ['Random Forest', 'Gradient Boosting', 'Logistic Regression', 'SVM']:
    print(f"  → {name}...", end=' ', flush=True)
    perm = permutation_importance(
        trained_models[name], X_sc, y.values,
        n_repeats=15, scoring='roc_auc',
        random_state=42, n_jobs=-1
    )
    perm_results[name] = perm
    print(f"done  (top feat: {feat_labels[np.argmax(perm.importances_mean)]})")

print("\n✅ Permutation importance computed!")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Permutation Importance (ROC-AUC Drop per Feature shuffle)',
             fontsize=15, fontweight='bold')
axes = axes.flatten()

for i, (name, perm) in enumerate(perm_results.items()):
    idx    = np.argsort(perm.importances_mean)
    labels = [feat_labels[j] for j in idx]
    means  = perm.importances_mean[idx]
    stds   = perm.importances_std[idx]

    # Boxplot style using bar + errorbar
    colors_pi = [C['red'] if m < 0 else MODEL_COLORS[name] for m in means]
    axes[i].barh(labels, means, xerr=stds, color=colors_pi, alpha=0.85, capsize=4,
                  error_kw=dict(ecolor='white', capthick=1.5))
    axes[i].axvline(0, color='white', linewidth=1.5)
    axes[i].set_title(f'{name}\nPermutation Importance (AUC drop)')
    axes[i].set_xlabel('Mean AUC decrease when feature shuffled')

plt.tight_layout()
plt.savefig('plots/13_permutation_importance.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print("✅ Saved: plots/13_permutation_importance.png")


## 14 · Individual Prediction Explanation — SHAP-Equivalent

> **Local Feature Contribution** (SHAP-like):
> For a single passenger, we measure how each feature **pushes the probability up or down**
> compared to the baseline (average passenger).
> Formula: `contribution[i] = model(x) − model(x with feature_i replaced by mean)`


In [ ]:
def explain_prediction(model, X_sc, X_df, sample_idx, feat_labels, baseline_prob=None):
    """
    Compute feature contributions for a single prediction.
    Approximation: contribution_i = P(x) - P(x with feature_i replaced by mean)
    """
    sample     = X_sc[sample_idx:sample_idx+1].copy()
    sample_pred = model.predict_proba(sample)[0, 1]

    if baseline_prob is None:
        baseline_prob = model.predict_proba(X_sc).mean(axis=0)[1]

    contributions = []
    for i in range(X_sc.shape[1]):
        x_mod    = sample.copy()
        x_mod[0, i] = X_sc[:, i].mean()  # replace with mean
        pred_without = model.predict_proba(x_mod)[0, 1]
        contributions.append(sample_pred - pred_without)

    return np.array(contributions), sample_pred, baseline_prob

# ── Pick 3 interesting passengers ─────────────────────────────────
# Survivor - 1st class female
mask_surv = (eng_df['Sex'] == 'female') & (eng_df['Pclass'] == 1) & (eng_df['Survived'] == 1)
idx_surv  = eng_df[mask_surv].index[0]

# Non-survivor - 3rd class male
mask_dead = (eng_df['Sex'] == 'male') & (eng_df['Pclass'] == 3) & (eng_df['Survived'] == 0)
idx_dead  = eng_df[mask_dead].index[0]

# Ambiguous - 2nd class male survivor (surprising)
mask_amb  = (eng_df['Sex'] == 'male') & (eng_df['Pclass'] == 2) & (eng_df['Survived'] == 1)
idx_amb   = eng_df[mask_amb].index[0]

best_model = trained_models[best_model_name]
baseline   = best_model.predict_proba(X_sc).mean(axis=0)[1]

for idx, desc in [(idx_surv, '1st Class Female — Survived'),
                   (idx_dead, '3rd Class Male   — Not Survived'),
                   (idx_amb,  '2nd Class Male   — Survived (Surprising)')]:
    contribs, pred, base = explain_prediction(
        best_model, X_sc, X, idx, feat_labels, baseline)
    row = eng_df.loc[idx]
    print(f"\n=== {desc} ===")
    print(f"  Name: {row['Name'][:45]}")
    print(f"  Pclass={row['Pclass']} | Sex={row['Sex']} | Age={row['Age']} | Fare=£{row['Fare']:.1f}")
    print(f"  Baseline P(survive) = {base*100:.1f}%")
    print(f"  Model prediction    = {pred*100:.1f}%")
    print(f"  Actual outcome      = {'SURVIVED' if row['Survived']==1 else 'NOT SURVIVED'}")
    top_pos = np.argsort(contribs)[-3:][::-1]
    top_neg = np.argsort(contribs)[:3]
    print("  Top positive contributors:")
    for i in top_pos:
        if contribs[i] > 0.001:
            print(f"    + {feat_labels[i]:25s} {contribs[i]:+.3f}")
    print("  Top negative contributors:")
    for i in top_neg:
        if contribs[i] < -0.001:
            print(f"    - {feat_labels[i]:25s} {contribs[i]:+.3f}")

print(f"\n✅ Using model: {best_model_name}")


In [ ]:
# Visualize explanations for all 3 passengers
fig, axes = plt.subplots(1, 3, figsize=(22, 8))
fig.suptitle(f'Individual Prediction Explanation — {best_model_name}',
             fontsize=15, fontweight='bold')

passengers = [
    (idx_surv, '1st Class Female\n(Survived)', C['green']),
    (idx_dead, '3rd Class Male\n(Not Survived)', C['red']),
    (idx_amb,  '2nd Class Male\n(Survived — Surprising)', C['yellow']),
]

for ax, (idx, title, c_accent) in zip(axes, passengers):
    contribs, pred, base = explain_prediction(
        best_model, X_sc, X, idx, feat_labels, baseline)

    # Sort by magnitude
    order   = np.argsort(contribs)
    labels  = [feat_labels[i] for i in order]
    vals    = contribs[order]
    colors  = [C['green'] if v >= 0 else C['red'] for v in vals]

    ax.barh(labels, vals, color=colors, alpha=0.88, edgecolor='none')
    ax.axvline(0, color='white', linewidth=1.5)

    # Annotate prediction
    verdict = 'SURVIVED' if pred >= 0.5 else 'NOT SURVIVED'
    v_color = C['green'] if pred >= 0.5 else C['red']
    ax.set_title(
        f"{title}\nBaseline:{base*100:.1f}% → Pred:{pred*100:.1f}% → {verdict}",
        fontsize=10, color=v_color
    )
    ax.set_xlabel('Feature Contribution (Probability change)')

    # Grid
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('plots/14_individual_explanation.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print("✅ Saved: plots/14_individual_explanation.png")


## 15 · Learning Curves — Bias vs Variance

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(24, 10))
fig.suptitle('Learning Curves — Bias-Variance Analysis (ROC-AUC)', fontsize=15,
             fontweight='bold')
axes = axes.flatten()

train_sizes_abs = np.linspace(0.1, 1.0, 10)

for i, (name, model) in enumerate([(n, get_models()[n]) for n in names_sorted]):
    tr_sizes, tr_scores, val_scores = learning_curve(
        model, X_sc, y.values, cv=cv,
        train_sizes=train_sizes_abs, scoring='roc_auc', n_jobs=-1
    )
    color = MODEL_COLORS[name]

    axes[i].plot(tr_sizes, tr_scores.mean(axis=1), 'o--',
                  color=C['gray'], linewidth=2, markersize=6, label='Train')
    axes[i].plot(tr_sizes, val_scores.mean(axis=1), 'D-',
                  color=color, linewidth=2.5, markersize=8, label='CV')
    axes[i].fill_between(tr_sizes,
        val_scores.mean(axis=1) - val_scores.std(axis=1),
        val_scores.mean(axis=1) + val_scores.std(axis=1),
        alpha=0.15, color=color)
    axes[i].fill_between(tr_sizes,
        tr_scores.mean(axis=1) - tr_scores.std(axis=1),
        tr_scores.mean(axis=1) + tr_scores.std(axis=1),
        alpha=0.1, color=C['gray'])

    gap = tr_scores.mean(axis=1)[-1] - val_scores.mean(axis=1)[-1]
    verdict = '✅' if gap < 0.05 else ('⚠' if gap < 0.12 else '❌')
    axes[i].set_title(f'{name}\n{verdict} Gap={gap*100:.1f}%', fontsize=10)
    axes[i].set_ylim(0.4, 1.05)
    axes[i].set_xlabel('Training Size')
    axes[i].set_ylabel('ROC-AUC')
    axes[i].legend(fontsize=8)

axes[-1].axis('off')
plt.tight_layout()
plt.savefig('plots/15_learning_curves.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print("✅ Saved: plots/15_learning_curves.png")


## 16 · Decision Tree — Visual Explanation

In [ ]:
# Train a shallow, interpretable tree
dt_viz = DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42)
dt_viz.fit(X_sc, y.values)

fig, ax = plt.subplots(figsize=(24, 10))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')

plot_tree(dt_viz, ax=ax,
          feature_names=[FEATURE_LABELS.get(f,f) for f in FEATURES],
          class_names=['Not Survived', 'Survived'],
          filled=True, rounded=True, fontsize=8,
          impurity=False, proportion=True,
          precision=2)
ax.set_title('Decision Tree (depth=4) — Interpretable Rule Path',
             fontsize=14, fontweight='bold', color='white', pad=15)

plt.tight_layout()
plt.savefig('plots/16_decision_tree.png', dpi=120, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()

# Text rules
rules = export_text(dt_viz,
    feature_names=[FEATURE_LABELS.get(f,f) for f in FEATURES])
print("Decision Rules (depth=4):")
print(rules[:2000], "..." if len(rules) > 2000 else "")


## 17 · Population-Level Feature Contributions

In [ ]:
# Compute feature contributions for all passengers
print(f"Computing contributions for all {len(X_sc)} passengers...")

n_sample = min(200, len(X_sc))  # subsample for speed
sample_idx = np.random.RandomState(42).choice(len(X_sc), n_sample, replace=False)

all_contribs = np.zeros((n_sample, len(FEATURES)))
for k, idx in enumerate(sample_idx):
    contribs, _, _ = explain_prediction(
        best_model, X_sc, X, idx, feat_labels, baseline)
    all_contribs[k] = contribs

print(f"✅ Done! Shape: {all_contribs.shape}")

# Summary: mean absolute contribution per feature
mean_abs = np.abs(all_contribs).mean(axis=0)
mean_raw = all_contribs.mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(f'Population-Level Feature Contributions — {best_model_name}\n'
             f'(Based on {n_sample} passengers)', fontsize=14, fontweight='bold')

# Mean absolute
idx_sorted = np.argsort(mean_abs)
axes[0].barh([feat_labels[i] for i in idx_sorted], mean_abs[idx_sorted],
              color=C['blue'], alpha=0.85)
axes[0].set_title('Mean |Contribution| — Global Importance')
axes[0].set_xlabel('Mean absolute probability change')

# Positive/negative split
pos_avg = np.where(all_contribs > 0, all_contribs, 0).mean(axis=0)
neg_avg = np.where(all_contribs < 0, all_contribs, 0).mean(axis=0)
idx_s2  = np.argsort(mean_abs)
labels2 = [feat_labels[i] for i in idx_s2]
axes[1].barh(labels2, pos_avg[idx_s2], color=C['green'], alpha=0.85, label='Positive push')
axes[1].barh(labels2, neg_avg[idx_s2], color=C['red'],   alpha=0.85, label='Negative push')
axes[1].axvline(0, color='white', linewidth=1.5)
axes[1].set_title('Average Direction of Contribution')
axes[1].set_xlabel('Avg probability change')
axes[1].legend()

plt.tight_layout()
plt.savefig('plots/17_population_contributions.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print("✅ Saved: plots/17_population_contributions.png")

print("\n=== TOP 5 MOST IMPACTFUL FEATURES ===")
for rank, i in enumerate(np.argsort(mean_abs)[-5:][::-1], 1):
    print(f"  {rank}. {feat_labels[i]:25s}  |Contribution|={mean_abs[i]:.4f}")


## 18 · Export Trained Models

In [ ]:
import pickle, os

os.makedirs('models', exist_ok=True)

# Export bundle
bundle = {
    'models':         trained_models,
    'scaler':         scaler,
    'features':       FEATURES,
    'feature_labels': FEATURE_LABELS,
    'best_model':     best_model_name,
    'cv_results': {
        name: {
            'auc_mean': float(res['test_roc_auc'].mean()),
            'auc_std':  float(res['test_roc_auc'].std()),
            'acc_mean': float(res['test_accuracy'].mean()),
            'f1_mean':  float(res['test_f1'].mean()),
        }
        for name, res in cv_results.items()
    }
}

with open('models/titanic_ml_bundle.pkl', 'wb') as f:
    pickle.dump(bundle, f)
print("✅ Saved: models/titanic_ml_bundle.pkl")

# Save best model separately
with open(f'models/best_model_{best_model_name.lower().replace(" ","_")}.pkl', 'wb') as f:
    pickle.dump({'model': trained_models[best_model_name], 'scaler': scaler,
                 'features': FEATURES}, f)
print(f"✅ Saved best model: {best_model_name}")

# Summary
print("\n" + "="*60)
print("FINAL MODEL RANKINGS (by ROC-AUC)")
print("="*60)
for i, (name, res) in enumerate(
    sorted(cv_results.items(), key=lambda kv: kv[1]['test_roc_auc'].mean(), reverse=True), 1
):
    star = " ★ BEST" if name == best_model_name else ""
    print(f"  {i}. {name:22s}  AUC={res['test_roc_auc'].mean()*100:.2f}%  "
          f"F1={res['test_f1'].mean()*100:.2f}%{star}")


## 19 · Research Summary & Conclusions

### 🏆 ML Pipeline Results

| Rank | Model | ROC-AUC | F1 | Verdict |
|------|-------|---------|-----|---------|
| 1 | **Best Model** (check output above) | ~85-88% | ~79-83% | 🥇 Deploy |
| 2 | Runner-up | ~84-87% | ~78-82% | ✅ Strong |
| 3-5 | Mid-tier | ~80-85% | ~74-79% | ✅ Solid |
| 6-7 | Baseline/Simple | ~75-80% | ~70-75% | ⚠️ Baseline |

---

### 🔍 Key Explainability Findings

1. **Sex (Female=1)** — Consistently #1 or #2 feature across ALL models. Women survived at 74% vs men at 19%.
2. **Title** — Encodes gender+status jointly. Miss/Mrs have the highest survival signal.
3. **Fare / Log_Fare / Fare_Per_Person** — Socioeconomic proxy. Higher fare → higher survival.
4. **Passenger Class** — Direct lifeboat access and evacuation priority effect.
5. **Has_Cabin** — Passengers with cabin records survived at ~67% vs ~30%.
6. **Age** — Non-linear. Children survived better; seniors did not.
7. **Family Size / Is_Alone** — Solo travelers and large families did worse.

---

### 📐 Individual Prediction Insight
- A **1st class female** passenger gets large **positive** contributions from Sex, Pclass, Fare → near 100% predicted survival
- A **3rd class male** gets large **negative** contributions from Sex, Pclass, low Fare → near 0% predicted survival
- **Counterintuitive cases** (2nd class male survivor) often have: young age, Master title, or small family

---

### 🛠️ Next Steps
- [ ] Kaggle test.csv prediction & submission
- [ ] Hyperparameter tuning with GridSearchCV
- [ ] SMOTE oversampling experiment
- [ ] Stacking ensemble (meta-learner)
- [ ] Deploy as REST API
